# HybridDocling OCR Benchmark

This notebook runs a full HybridDocling benchmark

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/buinguyenkhai/stock-report-agent-20251.git
%cd stock-report-agent-20251

In [ ]:
# Install system dependencies (Tesseract + Vietnamese language pack)
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-vie libtesseract-dev libleptonica-dev

# Install Python dependencies
%pip install -q -r "requirements.txt"
%pip install -q pymupdf
%pip install -q "docling[tesserocr]"
%pip install -q marker-pdf surya-ocr

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")

In [ ]:
# Sanity check imports
import sys
sys.path.insert(0, '.')
from evaluation.ocr_benchmark.page_level_benchmark import PageLevelBenchmark
from services.ocr.hybrid_pdf_pipeline import HybridPdfPipeline
print("Imports OK", PageLevelBenchmark, HybridPdfPipeline)

## 2) Run benchmark

Recommended for your goal: use `--financial-only` so the benchmark only evaluates statement/notes pages.

In [ ]:
# Run Hybrid-Docling on financial statement / notes pages only
!python -m evaluation.ocr_benchmark.page_level_benchmark \\
  --engine hybrid_docling \\
  --financial-only \\
  --save-outputs \\
  --output results/hybrid_docling_financial_only.json

## 3. Results Analysis

In [ ]:
import json

with open('results/hybrid_docling_full.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

def _pct(x):
    return 'N/A' if x is None else f"{x*100:.2f}%"

print('=', 'HybridDocling Full Benchmark Results', '=')
print(f"Total companies: {results.get('total_companies', 'N/A')}")
print(f"Total pages: {results.get('total_pages', 'N/A')}")
print(f"Successful pages: {results.get('successful_pages', 'N/A')}")
print('--- Overall Aggregated ---')
print('Word Recall:', _pct(results.get('overall_aggregated_word_recall')))
print('NumF1:', _pct(results.get('overall_aggregated_number_f1')))
print('--- Per-Page Averages ---')
print('Word Recall:', _pct(results.get('overall_avg_content_word_recall')))
print('NumF1:', _pct(results.get('overall_avg_number_f1')))
fa = results.get('overall_avg_format_agnostic_cer')
print('FA-CER:', 'N/A' if fa is None else f"{fa*100:.2f}%")

In [ ]:
!zip -r hybrid_docling_results.zip results